# AUV-Swarm-FL-RL | Kaggle Execution Hub

Notebook này được thiết kế để chạy trên Kaggle Notebook và thực thi pipeline theo thứ tự tối ưu thời gian:
1) Beta sensitivity (Fig 1, 2, 3)
2) Train PPO-only bootstrap model
3) Scheme comparison / ablation (Fig 4, 5, 6)
4) Train + plot baselines medium (Fig 7 - 100 rounds)
5) Train + plot baselines full (Fig 7 - 1000 rounds)
6) Zip kết quả vào /kaggle/working để tải về

## 1. Clone Source Code và Cài Đặt Môi Trường (Kaggle)
Cập nhật URL repository bên dưới về repo GitHub của bạn trước khi chạy cell.

In [1]:
import os
import shutil

# TODO: Cập nhật URL repo của bạn
REPO_URL = "https://github.com/ngnam1104/AUV-Swarm-RFL.git"
WORKDIR = "/kaggle/working"
REPO_DIR = os.path.join(WORKDIR, "AUV-Swarm-RFL")

# Xóa bản clone cũ nếu tồn tại
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

# Clone repo vào thư mục làm việc của Kaggle
!git clone $REPO_URL $REPO_DIR

# Di chuyển vào repo
os.chdir(REPO_DIR)

# Tạo thư mục log giống pipeline PowerShell
os.makedirs("results/logs", exist_ok=True)

# Cài đặt dependencies
!pip install -r requirements.txt

print("Kaggle environment is ready")
print("Current working directory:", os.getcwd())

Cloning into '/kaggle/working/AUV-Swarm-RFL'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 50 (delta 10), reused 44 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 84.84 KiB | 3.86 MiB/s, done.
Resolving deltas: 100% (10/10), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 94.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 52.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 116.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Bước 1 - Beta Sensitivity (Figure 1, 2, 3)
Chạy thí nghiệm độ nhạy của hệ số beta tương tự bước đầu tiên trong script PowerShell.

In [3]:
# !python -u scripts/eval_beta_sensitivity.py \
#   --rounds 50 \
#   --m-values 9 16 25 \
#   --out-dir results/beta_sensitivity

[INFO] Start beta sensitivity evaluation
[INFO] rounds=50, m_values=[9, 16, 25], betas=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
[INFO] early_stopping=OFF (fixed rounds)
[WARN] rounds=50 < 1000. This is a smoke run; curves may look flat or noisy.

[JOB START] M=9, beta=0.1
Failed to download (trying next):
HTTP Error 404: Not Found

100%|███████████████████████████| 9912422/9912422 [00:00<00:00, 52976659.12it/s]
Extracting /kaggle/working/AUV-Swarm-RFL/.data/MNIST/raw/train-images-idx3-ubyte.gz to /kaggle/working/AUV-Swarm-RFL/.data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found

100%|████████████████████████████████| 28881/28881 [00:00<00:00, 1577308.22it/s]
Extracting /kaggle/working/AUV-Swarm-RFL/.data/MNIST/raw/train-labels-idx1-ubyte.gz to /kaggle/working/AUV-Swarm-RFL/.data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found

100%|███████████████████████████| 1648877/1648877 [00:00<00:00, 14131685.42it/s]
Extracting /kaggle/working/A

## 3. Bước 2 - Train PPO-Only Bootstrap Model
Chỉ huấn luyện PPO để tạo model đầu vào cho thí nghiệm Figure 4, 5, 6 (nhanh hơn train đủ 4 thuật toán).

In [ ]:
!python -u scripts/train_baselines.py \
  --m 9 \
  --max-fl-rounds 200 \
  --episodes 500 \
  --eval-interval 5 \
  --algorithms ppo ddpg greedy random \
  --print-every-steps 10 \
  --out-dir results/fig_7 \
  --step-log-file results/logs/baselines_step_log.txt

## 4. Bước 3 - Plot Figure 7 
Vẽ biểu đồ Figure 7 cho kết quả medium (200 rounds).

In [ ]:
!python -u scripts/plot_fig_7.py \
  --input-dir results/fig_7 \
  --sigma 2.0 \
  --out-path results/fig_7/figure7.png

from IPython.display import Image, display
display(Image("results/fig_7/figure7.png"))

## 5. Bước 4 - Scheme Comparison / Ablation (Figure 4, 5, 6)
Chạy Figure 4, 5, 6 ngay sau khi đã có PPO bootstrap model.

In [ ]:
!python -u scripts/run_fig_4_5_6.py \
  --rounds 200 \
  --m-values 9 16 25 36 49 \
  --model-path results/fig_7/ppo_baseline_model \
  --lag-threshold 1e4 \
  --out-dir results/eval_schemes

## 6. Bước 5 - và Zip Kết Quả
Vẽ Figure 7 full, sau đó nén toàn bộ thư mục results vào /kaggle/working để tải về.

In [ ]:
ZIP_PATH = "/kaggle/working/experiment_results.zip"
!zip -r $ZIP_PATH results/

print("Zip file saved at:", ZIP_PATH)
print("Open the right panel in Kaggle (Output/Files) to download experiment_results.zip")